# Stream-Mind 데모 (Qwen3.5-0.8B, 4단계 모델)

모델은 끝나지 않는 스트림 하나 위에서 돈다. 메시지는 메신저 알림으로 들어오고, 모델은 `<think>` 속에서 도구를 부르며
(메신저로 답, 노트에 적기·찾기), 생각이 끝나면 쉰다. 한 세션의 대화는 같은 스트림에 이어지므로 앞에서 한 말을 기억해야 한다.

**쓰는 법**: 오른쪽 위 **Edit**(또는 Copy & Edit)으로 편집 화면을 열고, 설정에서 가속기가 **GPU T4**인지 확인한 뒤 위에서부터 셀을 실행한다.

- 흐린 글씨: 관측(알림, 메시지 내용, 실행 기록) · 파란 글씨: 모델이 쓴 사고와 도구 호출
- 가상 시계로 돈다(사람의 타이핑 속도로 메시지가 도착하고, 사고 토큰마다 시간이 흐른다). 실제 시간으로 돌리려면 `Session(model, realtime=True)`.


In [ ]:
# 1. 소스 설치 (입력 데이터셋 irukidev/stream-mind-src, 1분 안팎)
import subprocess, sys, shutil
from pathlib import Path
src = next(p.parent for p in Path("/kaggle/input").rglob("pyproject.toml") if (p.parent / "src/stream_mind").is_dir())
if not Path("/tmp/stream-mind").exists():
    shutil.copytree(src, "/tmp/stream-mind")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.19.0"], check=True)
sys.path.insert(0, "/tmp/stream-mind/src")
ckpt = next(Path("/kaggle/input").rglob("stage4_stream.pt"))
print("소스", src, "\n체크포인트", ckpt)

In [ ]:
# 2. 모델 불러오기 (Qwen3.5-0.8B 본체 + 스트림 어댑터, 1~2분)
import os
os.environ["STREAM_MIND_DEVICE"] = "cuda"
from stream_mind.demo import Session, chat, play, load
model = load(str(ckpt), "cuda")
print(model.device, f"{model.num_parameters():,} 매개변수")

## 예시 대화
같은 세션(스트림)에 메시지를 차례로 보낸다. 세 번째 질문은 첫 메시지를 기억해야 답할 수 있다.

In [ ]:
s = Session(model)
for msg in ["안녕, 내 이름은 지아야", "계산: 37*24", "내 이름이 뭐야?",
            "기억해: 회의실는 B204", "물의 끓는점은 몇 도야?", "회의실가 뭐였지?"]:
    print(f"\n나 ▶ {msg}")
    print("모델 ▶", s.say(msg))

## 직접 대화하기
아래 셀을 실행하면 입력 칸이 뜬다. `/think`로 사고 보기를 켜고 끄고, `/reset`으로 새 세션, `/quit`으로 끝낸다.

수학은 "…를 풀어 줘"처럼, 객관식은 보기를 `(A) … (B) …`로 주고 "기호로 답하세요"를 붙이면 학습한 형식과 같다.

In [ ]:
chat(model)

## 게임 한 판
멈추지 않는 게임. 압박 = 끝까지 생각하는 데 걸리는 시간 / 프레임 간격. 1보다 크면 생각을 다 하기 전에 프레임이 넘어간다.

게임은 이 모델의 약한 부분이다(관문 2·5 미달, `docs/qwen.md`).

In [ ]:
play(model, "freeway-lite", pressure=1.0, frames=12, seed=3)